# 5.13 · 多类与多标签 / Multi-class & Multi-label Classification

> **课程定位 / Where this fits**
> 5.2 用 softmax 原生多分类, 5.5 SVM 用 OvO。这一课**系统化**: 怎样把只会二分类的模型(SVM、逻辑回归)拼成多分类(OvR/OvO), 以及一个根本不同的问题——**多标签**(一个样本可同时属多个类, 如一篇文章既是"科技"又是"政治")。
> Systematic treatment: turning binary learners into multi-class (OvR/OvO), and the distinct multi-label problem where a sample can carry several labels at once.

> 💡 **面试相关 / Interview-relevant**
> - "OvR vs OvO: 分类器个数 / 优缺点" ★★★★★
> - "多分类 vs 多标签 vs 多输出 区别" ★★★★★
> - "多标签如何评估(Hamming/micro-macro F1)" ★★★★
> - "分类器链 vs binary relevance" ★★★★
> - "micro vs macro 平均区别" ★★★★

---

## 学习目标 / Learning Objectives
1. **OvR / OvO** 把二分类器拼成多分类(个数、代价、何时用)。
2. **多类 vs 多标签 vs 多输出**的概念区分。
3. 多标签策略: **binary relevance** 与 **classifier chain**。
4. 多类/多标签的**评估**: micro/macro F1, Hamming loss。

## 目录 / TOC
1. [OvR vs OvO ⭐](#1)
2. [📰 数据: 20 Newsgroups](#2)
3. [多类评估: micro/macro ⭐](#3)
4. [多标签: 概念与策略 ⭐](#4)
5. [分类器链 + 多标签评估](#5)
6. [小结](#6)


<a id="1"></a>
## 1. OvR vs OvO ⭐ / One-vs-Rest vs One-vs-One

很多模型(SVM、逻辑回归)本质二分类。两种拼成 K 类的方法:

| | One-vs-Rest (OvR) | One-vs-One (OvO) |
|---|---|---|
| 分类器数 | **K** 个("类k vs 其余") | **K(K-1)/2** 个(每对一个) |
| 每个训练用 | 全部数据 | 只用两类的数据 |
| 预测 | 取分数最高的类 | 投票最多的类 |
| 适合 | K 大、训练慢的模型 | 单个训练成本对 n 超线性的模型(如 SVM) |
| 默认 | 逻辑回归/LinearSVC | SVC(核 SVM) |

**直觉**: OvR 分类器少但每个要分"一类 vs 一大堆"(可能不平衡); OvO 分类器多但每个简单(只分两类)、且每个数据少。SVM 训练 $O(n^2)$, OvO 每个只用 $2n/K$ 数据反而总体更省, 故 SVC 默认 OvO。


<a id="2"></a>
## 2. 数据: 20 Newsgroups / The 20 Newsgroups Dataset

经典文本多分类: ~2 万篇新闻组帖子, 分属 20 个主题。这里取 **4 个主题**的子集(计算机图形/曲棍球/太空/中东政治), 去掉头尾引用避免泄漏, 用 TF-IDF(3.7)向量化。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
sns.set_theme(style="whitegrid")

cats = ['comp.graphics', 'rec.sport.hockey', 'sci.space', 'talk.politics.mideast']
train = fetch_20newsgroups(subset='train', categories=cats, remove=('headers','footers','quotes'), random_state=0)
test  = fetch_20newsgroups(subset='test',  categories=cats, remove=('headers','footers','quotes'), random_state=0)
print(f"20 Newsgroups 子集: train {len(train.data)} / test {len(test.data)} 篇, {len(cats)} 类")
print("类别:", train.target_names)
print("\n样例(sci.space):", train.data[ list(train.target).index(2) ][:160].replace(chr(10),' '), "...")

vec = TfidfVectorizer(max_features=5000, stop_words='english')
Xtr = vec.fit_transform(train.data); Xte = vec.transform(test.data)
ytr, yte = train.target, test.target
print(f"\nTF-IDF 矩阵: {Xtr.shape}")


<a id="3"></a>
## 3. 多类评估: micro/macro ⭐ / Multiclass Metrics


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

base = LinearSVC()
ovr = OneVsRestClassifier(base).fit(Xtr, ytr)
ovo = OneVsOneClassifier(LinearSVC()).fit(Xtr, ytr)
print(f"OvR: {len(ovr.estimators_)} 个分类器, test 准确率 {accuracy_score(yte, ovr.predict(Xte)):.3f}")
print(f"OvO: {len(ovo.estimators_)} 个分类器, test 准确率 {accuracy_score(yte, ovo.predict(Xte)):.3f}")
print(f"(OvO 个数 = C(4,2) = 6)")


**micro vs macro 平均**(面试常考)——把多类的 P/R/F1 汇总成一个数:
- **macro**: 每类各算 F1 再**简单平均** → 每类等权, **少数类影响大**。
- **micro**: 把所有类的 TP/FP/FN **汇总**再算 → 每**样本**等权, **被多数类主导**。
- **weighted**: 按各类样本数加权的 macro。


In [ ]:
pred = ovr.predict(Xte)
for avg in ["micro", "macro", "weighted"]:
    print(f"F1 ({avg:<8}): {f1_score(yte, pred, average=avg):.3f}")
print("\n本子集各类样本数较均衡, 三者接近; 不平衡时 macro 会显著低于 micro")
print("\n逐类报告:")
print(classification_report(yte, pred, target_names=[c.split('.')[-1] for c in cats], digits=3))


<a id="4"></a>
## 4. 多标签: 概念与策略 ⭐ / Multi-label

**概念区分**(高频混淆点):
- **多类(multi-class)**: 一个样本属**恰好一个**类(Iris→一个品种)。标签互斥。
- **多标签(multi-label)**: 一个样本可属**多个**类(一篇文章同时是"太空"+"政治")。标签不互斥。
- **多输出(multi-output)**: 预测多个**独立的目标变量**(每个可多类)。

**多标签策略**:
1. **Binary Relevance**: 每个标签训一个独立二分类器(忽略标签间相关性)。
2. **Classifier Chain**: 串联——后面的分类器把前面预测的标签也当特征, **捕捉标签相关性**。


In [ ]:
# 构造一个多标签问题(合成, 标签间有相关性) / synthetic multi-label
from sklearn.datasets import make_multilabel_classification
from sklearn.model_selection import train_test_split
Xm, Ym = make_multilabel_classification(n_samples=2000, n_features=30, n_classes=5,
                                        n_labels=2, random_state=0)
Xm_tr, Xm_te, Ym_tr, Ym_te = train_test_split(Xm, Ym, test_size=0.3, random_state=0)
print(f"多标签数据: X{Xm.shape}, Y{Ym.shape} (5 个标签, 每样本平均 {Ym.sum(1).mean():.1f} 个标签)")
print("标签矩阵示例(每行可有多个1):"); print(Ym_tr[:4])


<a id="5"></a>
## 5. 分类器链 + 多标签评估 / Chains & Metrics

多标签评估指标:
- **Hamming loss**: 预测错的标签位比例(越低越好), 标签级。
- **subset accuracy(精确匹配)**: 一个样本**所有标签全对**才算对(最严格)。
- **micro/macro F1**: 同上, 跨所有标签汇总。


In [ ]:
from sklearn.multioutput import ClassifierChain
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import hamming_loss, accuracy_score, f1_score

# Binary Relevance (独立) vs Classifier Chain (链)
br = OneVsRestClassifier(LogisticRegression(max_iter=1000)).fit(Xm_tr, Ym_tr)
chain = ClassifierChain(LogisticRegression(max_iter=1000), order="random", random_state=0).fit(Xm_tr, Ym_tr)

for name, model in [("Binary Relevance", br), ("Classifier Chain", chain)]:
    P = model.predict(Xm_te)
    P = (P > 0.5).astype(int) if P.dtype != int else P
    print(f"{name}:")
    print(f"   Hamming loss   {hamming_loss(Ym_te, P):.3f} (越低越好)")
    print(f"   subset 精确匹配 {accuracy_score(Ym_te, P):.3f} (全标签对才算对)")
    print(f"   micro-F1       {f1_score(Ym_te, P, average='micro'):.3f}")
print("\nClassifier Chain 利用标签相关性, subset 精确匹配通常优于独立的 Binary Relevance")


<a id="6"></a>
## 6. 小结 / Summary

```
OvR: K 个"一类vs其余", 每个用全数据; OvO: K(K-1)/2 个两两分类, 每个用两类数据
  SVC 默认 OvO(训练超线性时更省), LR/LinearSVC 默认 OvR
多类(恰一类) vs 多标签(可多类) vs 多输出(多个目标)
多标签: Binary Relevance(独立) vs Classifier Chain(串联, 用标签相关性)
评估: micro(样本等权,多数主导) vs macro(类等权,少数影响大) vs weighted
  多标签另有 Hamming loss(标签级) 与 subset accuracy(全对才算)
```

### 💡 面试速查
1. **OvR(K个) vs OvO(K(K-1)/2个)**: OvO 每个简单但多, SVM 用 OvO
2. **多类(互斥) vs 多标签(可共存) vs 多输出(多目标)**
3. **多标签**: binary relevance 忽略相关性, classifier chain 捕捉它
4. **micro(多数主导) vs macro(少数等权)**: 不平衡看 macro
5. 多标签评估: Hamming loss + subset accuracy + micro/macro F1

### 下一节
**5.14 不平衡分类**——前面假设类别大致均衡。欺诈/罕见病等正类只占 1% 时, 准确率失效。重采样(SMOTE)、类权重、阈值移动、代价敏感学习。
